<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/EXPORT/EXP_SUNAT_DUA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exportaciones por DUA · SUNAT (Aduanet)

Detalle de exportaciones **por empresa**, para los años que Infotrade no tiene (antes de 2021) o para revisar cualquier año.

1. **Bloque 1:** instala y define funciones.
2. **Bloque 2:** pega los RUC (o el ID de Drive de tu base de empresas) y elige los años.
3. **Bloque 3 · Fase A (sola, sin captcha):** «Consulta por exportador» → FOB por mes, agente, aduana y país → «LISTAR» → números de DUA. Deja el índice de DUA en Drive.
4. **Bloque 4 · Fase B (con captcha; ejecuta antes el Bloque 3, que conecta Drive y carga el índice aunque no haya nada pendiente):** abre cada DUA del índice. Tú escribes el captcha y el notebook baja la cabecera (RUC, dirección con distrito/provincia/departamento, FOB, pesos, agente, transporte) y **todas las series** (partida, FOB, pesos, cantidad y unidad, descripción comercial y especie).

El captcha **no** se resuelve automáticamente: SUNAT lo pone para que lo escriba una persona. Todo lo demás es automático.

Todo el HTML original queda en Drive (`SUNAT_DUA/SUNAT_DUA_HTML`, en ZIP) para poder releerlo si el lector necesita ajustes. Detalle en `EXPORT/exportaciones_scraper_infotrade.md`, §9.

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN Y FUNCIONES (ejecutar una vez por sesión) ═══
!pip install requests pandas beautifulsoup4 lxml ipywidgets google-api-python-client openpyxl --quiet
import os, io, re, json, time, random, zipfile, shutil
from datetime import datetime
from urllib.parse import urljoin, urlparse, parse_qs, urlencode
import requests
import pandas as pd
import ipywidgets as w
from bs4 import BeautifulSoup
from IPython.display import display, Image

BASE = "http://www.aduanet.gob.pe"
URL_EXPORTADOR = BASE + "/cl-ad-itconsultadwh/ieITS01Alias"   # consulta por exportador (sin captcha)
URL_DUA = BASE + "/servlet/SgCDUI2"                             # consulta de una DUA (con captcha)
REGIMEN = "40"                                                  # exportación definitiva

sesion = requests.Session()
sesion.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                       "Accept-Language": "es-PE,es;q=0.9"})

def texto(resp):
    """HTML de la respuesta. SUNAT manda UTF-8 sin declararlo (en el navegador se ve «CÃ³digo»)."""
    b = resp.content
    try:
        return b.decode("utf-8")
    except UnicodeDecodeError:
        return b.decode("latin-1")

def pedir(metodo, url, reintentos=4, **kw):
    """GET/POST con reintentos y backoff. -> (html, respuesta) o (None, error)."""
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion.request(metodo, url, timeout=kw.pop("timeout", 120), **kw)
            if r.status_code == 200:
                return texto(r), r
            ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(60, 3 * 2 ** i + random.random()))
    return None, ultimo

def pausa():
    time.sleep(random.uniform(*PAUSA))

def limpio(t):
    return " ".join(str(t or "").replace("\xa0", " ").split())

def numero(t):
    """'4,868.66' -> 4868.66 ; '17.666 - M3' -> 17.666 ; '' -> None"""
    m = re.search(r"-?[\d,]*\.?\d+", str(t or "").replace(" ", ""))
    if not m:
        return None
    try:
        return float(m.group(0).replace(",", ""))
    except ValueError:
        return None

def campos_form(form):
    """Valores por defecto de un <form>: inputs (salvo botones), selects y textareas."""
    datos = {}
    for inp in form.find_all("input"):
        n, t = inp.get("name"), (inp.get("type") or "text").lower()
        if not n or t in ("button", "submit", "reset", "image", "file"):
            continue
        if t in ("checkbox", "radio"):
            if inp.has_attr("checked"):
                datos[n] = inp.get("value", "on")
            continue
        datos[n] = inp.get("value", "")
    for sel in form.find_all("select"):
        n = sel.get("name")
        if not n:
            continue
        op = sel.find("option", selected=True) or sel.find("option")
        datos[n] = (op.get("value", op.get_text()) if op else "").strip()
    for ta in form.find_all("textarea"):
        if ta.get("name"):
            datos[ta["name"]] = ta.get_text()
    return datos

def enviar_form(form, datos, url_pagina):
    accion = urljoin(url_pagina, form.get("action") or url_pagina)
    if (form.get("method") or "get").lower() == "post":
        return pedir("POST", accion, data=datos, headers={"Referer": url_pagina})
    return pedir("GET", accion, params=datos, headers={"Referer": url_pagina})

def opciones(sel):
    return [((o.get("value") if o.get("value") is not None else o.get_text()).strip(), limpio(o.get_text()))
            for o in sel.find_all("option")]

def links_con(html, clave, url_base):
    """URLs (href u onclick) que contienen `clave`, con su fila <tr> -> [(url, [celdas])]."""
    soup, out, vistos = BeautifulSoup(html, "lxml"), [], set()
    for a in soup.find_all(["a", "area", "input"]):
        cand = " ".join(filter(None, [a.get("href"), a.get("onclick")]))
        m = re.search(r"([\w/\.\-]*\?[^'\"<>\s]*" + re.escape(clave) + r"[^'\"<>\s]*)", cand)
        if not m:
            continue
        url = urljoin(url_base, m.group(1).replace("&amp;", "&"))
        if url in vistos:
            continue
        vistos.add(url)
        tr = a.find_parent("tr")
        out.append((url, [limpio(td.get_text(" ")) for td in tr.find_all(["td", "th"], recursive=False)] if tr else []))
    return out

def paginacion(html):
    """'1 a 6 de 6' -> (6, 6): filas mostradas hasta y total."""
    m = re.search(r"(\d+)\s+a\s+(\d+)\s+de\s+(\d+)", BeautifulSoup(html, "lxml").get_text(" "))
    return (int(m.group(2)), int(m.group(3))) if m else (None, None)

def siguiente_pagina(html, url_base):
    soup = BeautifulSoup(html, "lxml")
    for a in soup.find_all("a"):
        t = limpio(a.get_text()).lower()
        if t in ("siguiente", ">", ">>", "sig.", "sgte", "siguiente >") and a.get("href") and not a["href"].startswith("javascript"):
            return urljoin(url_base, a["href"])
    return None

# ── Lectura de las fichas de la DUA (rótulo arriba, valor en la fila de abajo) ──
RE_ROTULO = re.compile(r"^\d{1,2}(\.\d{1,2})+\s+\S")
ROTULOS_SUELTOS = {"Aduana", "Código", "Nº Orden", "Destinación", "Modalidad", "Tipo Despacho",
                   "Nº DUA Prov.", "Nº Orden de Embarque"}

def _span(td, atr):
    v = str(td.get(atr, 1)).strip()
    return max(1, int(v)) if v.isdigit() else 1

def filas_grilla(tabla):
    """Filas de una tabla como [(col_inicio, col_fin, texto)], respetando colspan y rowspan."""
    filas, ocupado = [], {}                       # ocupado[(fila, col)] = True por un rowspan de arriba
    trs = [tr for tr in tabla.find_all("tr") if tr.find_parent("table") is tabla]
    for i, tr in enumerate(trs):
        col, fila = 0, []
        for td in tr.find_all(["td", "th"], recursive=False):
            while ocupado.get((i, col)):
                col += 1
            cs, rs = _span(td, "colspan"), _span(td, "rowspan")
            # el texto de las tablas anidadas se lee aparte (cada tabla tiene su propia grilla)
            t = limpio(" ".join(s for s in td.find_all(string=True) if s.find_parent("table") is tabla))
            fila.append((col, col + cs, t))
            for r in range(1, rs):
                for c in range(col, col + cs):
                    ocupado[(i + r, c)] = True
            col += cs
        filas.append(fila)
    return filas

def leer_ficha(html):
    """{rótulo: valor} de una página de DUA o de serie (todas las tablas)."""
    soup, out = BeautifulSoup(html, "lxml"), {}
    for tabla in soup.find_all("table"):
        filas = filas_grilla(tabla)
        for i, fila in enumerate(filas[:-1]):
            abajo = filas[i + 1]
            for c0, c1, t in fila:
                if not t or not (RE_ROTULO.match(t) or t in ROTULOS_SUELTOS or t.startswith("Total ")) or t in out:
                    continue
                val = [tt for a0, a1, tt in abajo if a0 < c1 and a1 > c0]
                if val and not RE_ROTULO.match(val[0]):
                    out[t] = val[0]
    texto_plano = limpio(soup.get_text(" "))
    for clave, patron in [("Nº Declaración", r"N\S{0,2}\s*Declaraci\S*n\s*:\s*(\d+)"),
                          ("Fecha Numeración", r"Fecha\s+Numeraci\S*n\s*:\s*(\d{2}/\d{2}/\d{4})"),
                          ("Sujeto a", r"Sujeto\s+a\s*:\s*([A-ZÁÉÍÓÚ]+)")]:
        m = re.search(patron, texto_plano)
        if m:
            out[clave] = m.group(1)
    return out

def valor(ficha, prefijo):
    """Valor del primer rótulo que empieza por `prefijo` (p. ej. '7.30')."""
    for k, v in ficha.items():
        if k == prefijo or k.startswith(prefijo + " "):
            return v
    return ""

def descripcion_serie(html):
    """Líneas '1. MADERA ASERRADA…', '2. ESPECIE: …' del rótulo 7.35."""
    lineas = {}
    for td in BeautifulSoup(html, "lxml").find_all("td"):
        if td.find("table"):
            continue
        m = re.match(r"^([1-5])\.\s*(.*)$", limpio(td.get_text(" ")))
        if m and m.group(1) not in lineas and not RE_ROTULO.match(limpio(td.get_text(" "))):
            lineas[m.group(1)] = m.group(2)
    return [lineas.get(str(i), "") for i in range(1, 6)]

print("✅ Librerías y funciones listas")

In [ ]:
# ═══ BLOQUE 2 · SELECTORES (elige y luego ejecuta el Bloque 3) ═══
# Empresas: pega los RUC (uno por línea; se aceptan "RUC  RAZÓN SOCIAL") o indica el ID de Drive
# de tu base de empresas (Excel o CSV: se usa la columna que tiene RUC de 11 dígitos).
txt_rucs = w.Textarea(placeholder="20493258274\n20493285590 SICO MADERAS…",
                      layout=w.Layout(width="520px", height="140px"))
txt_base = w.Text(placeholder="ID de Drive del Excel/CSV de empresas (opcional)",
                  layout=w.Layout(width="520px"))
sel_anio_ini = w.Dropdown(options=list(range(1999, datetime.now().year + 1)), value=2005)
sel_anio_fin = w.Dropdown(options=list(range(1999, datetime.now().year + 1)), value=2020)
chk_diag = w.Checkbox(value=True, description="Modo diagnóstico: guardar en Drive el HTML de las primeras páginas",
                      layout=w.Layout(width="620px"), style={"description_width": "initial"})
txt_max_duas = w.BoundedIntText(value=200, min=1, max=100000, description="DUA por corrida (Bloque 4):",
                                style={"description_width": "initial"})
txt_plantilla = w.Text(placeholder="(opcional) URL completa de una serie, copiada de la ventana «Serie de la Declaración»",
                       layout=w.Layout(width="720px"))

def caja(titulo, *hijos):
    return w.VBox([w.HTML(f"<b>{titulo}</b>"), *hijos], layout=w.Layout(margin="0 0 10px 0"))

display(caja("Empresas (RUC)", txt_rucs, w.HTML("o"), txt_base),
        caja("Años (consulta por exportador, todo el año)", w.HBox([w.Label("Desde"), sel_anio_ini,
                                                                   w.Label("Hasta"), sel_anio_fin])),
        caja("Detalle de DUA (Bloque 4, con captcha)", txt_max_duas, txt_plantilla),
        chk_diag)

In [ ]:
# ═══ BLOQUE 3 · FASE A: EXPORTADOR → MESES → DUA (sin captcha, corre solo) ═══
# Por cada RUC y año: «Consulta por exportador» (todo el año, EXPORTACION) y luego «LISTAR» de cada fila
# (mes × agente × aduana × país) para obtener los números de DUA. Si se corta, vuelve a ejecutarlo:
# salta los RUC-año que ya están OK en el control.

# ── 1. CONFIGURACIÓN FIJA ────────────────────────────────────────────────────
DRIVE_FOLDER_ID = "1JJqcvUomccL-dCCNm5yovl-9TN5laKYi"           # DATA (se crean subcarpetas)
DRIVE_FOLDER_CONTROL_ID = "1gAO8-UjTV2bmBwMPgONBAYeVfpwXSWCY"   # CONTROL
PAUSA = (1.5, 3.0)                       # segundos aleatorios entre consultas (cortesía con SUNAT)
LOCAL = "/content/SUNAT_DUA"
os.makedirs(f"{LOCAL}/html", exist_ok=True)
HOY = datetime.now().strftime("%d/%m/%Y")
N_EXPORTADOR, N_INDICE = "SUNAT_EXPORTADOR_MES.csv", "SUNAT_DUA_INDICE.csv"
N_CABECERA, N_SERIES = "SUNAT_DUA_CABECERA.csv", "SUNAT_DUA_SERIES.csv"
N_CONTROL = "CONTROL SUNAT_DUA exportador-año.csv"

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import logging
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)

for fid in (DRIVE_FOLDER_ID, DRIVE_FOLDER_CONTROL_ID):
    try:
        c = drive.files().get(fileId=fid, fields="id,name,mimeType,capabilities(canAddChildren)",
                              supportsAllDrives=True).execute()
    except HttpError as e:
        raise PermissionError(f"Sin acceso a la carpeta de Drive {fid} (HTTP {e.resp.status}).") from e
    if not c.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(f"Sin permiso de Editor en '{c['name']}'.")
    print(f"☁️  Drive conectado -> carpeta: '{c['name']}'")

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def carpeta(nombre, padre):
    ya = archivos_en(padre)
    if nombre in ya:
        return ya[nombre]
    return drive.files().create(body={"name": nombre, "parents": [padre],
                                      "mimeType": "application/vnd.google-apps.folder"},
                                fields="id", supportsAllDrives=True).execute()["id"]

def subir(ruta_local, folder_id, existentes, mime="text/csv", reintentos=5):
    nombre = os.path.basename(ruta_local)
    for i in range(1, reintentos + 1):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [folder_id]},
                                                          media_body=media, fields="id",
                                                          supportsAllDrives=True).execute()["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/{reintentos}: {e}")
            time.sleep(2 ** i)
    print(f"   ❌ No se pudo subir {nombre}. Queda en {ruta_local}")
    return False

def bajar(file_id):
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
    hecho = False
    while not hecho:
        _, hecho = dl.next_chunk()
    buf.seek(0)
    return buf

F_DATOS = carpeta("SUNAT_DUA", DRIVE_FOLDER_ID)
F_HTML = carpeta("SUNAT_DUA_HTML", F_DATOS)
datos_drive, control_drive, html_drive = archivos_en(F_DATOS), archivos_en(DRIVE_FOLDER_CONTROL_ID), archivos_en(F_HTML)

def leer_previo(nombre, existentes, cols):
    if nombre in existentes:
        df = pd.read_csv(bajar(existentes[nombre]), dtype=str, encoding="utf-8-sig").fillna("")
        for c in cols:
            if c not in df.columns:
                df[c] = ""
        return df
    return pd.DataFrame(columns=cols)

def guardar(df, nombre, folder_id, existentes):
    ruta = f"{LOCAL}/{nombre}"
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    subir(ruta, folder_id, existentes)

def guardar_html(nombre, html):
    with open(f"{LOCAL}/html/{nombre}", "w", encoding="utf-8") as f:
        f.write(html or "")

def subir_html(etiqueta):
    """Comprime los HTML acumulados en un ZIP, lo sube a SUNAT_DUA_HTML y vacía la carpeta local."""
    archivos = os.listdir(f"{LOCAL}/html")
    if not archivos:
        return
    z = f"{LOCAL}/HTML {etiqueta} {datetime.now():%Y%m%d-%H%M%S}.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        for a in archivos:
            zf.write(f"{LOCAL}/html/{a}", a)
    if subir(z, F_HTML, html_drive, mime="application/zip"):
        shutil.rmtree(f"{LOCAL}/html"); os.makedirs(f"{LOCAL}/html"); os.remove(z)

COLS_EXP = ["RUC", "EXPORTADOR", "AÑO", "MES", "MES_TXT", "AGENTE", "ADUANA", "PAIS", "FOB_USD",
            "ADUANA_COD", "AGENTE_COD", "PAIS_COD", "URL_LISTAR", "DUAS_LISTADAS", "DUAS_TOTAL", "FECHA_EXTRACCION"]
COLS_IND = ["DUA_ID", "RUC", "AÑO", "MES", "ADUANA_COD", "AGENTE_COD", "PAIS_COD", "ANIO_DUA", "NUMERO",
            "FECHA_EMBARQUE", "CANAL", "ESTADO_DETALLE", "N_SERIES", "FOB_DUA", "FOB_SERIES", "NOTA",
            "FECHA_DETALLE", "FECHA_EXTRACCION"]
COLS_CTRL = ["RUC", "AÑO", "FILAS_MES", "FOB_USD_MES", "LISTADOS_OK", "DUAS", "ESTADO", "NOTA", "FECHA_EXTRACCION"]
exp_df = leer_previo(N_EXPORTADOR, datos_drive, COLS_EXP)
ind_df = leer_previo(N_INDICE, datos_drive, COLS_IND)
ctrl_df = leer_previo(N_CONTROL, control_drive, COLS_CTRL)
print(f"📄 Previo: {len(exp_df)} filas exportador-mes, {len(ind_df)} DUA en el índice, {len(ctrl_df)} RUC-año en el control")

# ── 3. EMPRESAS ──────────────────────────────────────────────────────────────
def rucs_de_texto(t):
    return re.findall(r"(?<!\d)(?:10|15|17|20)\d{9}(?!\d)", t)

rucs = rucs_de_texto(txt_rucs.value)
if txt_base.value.strip():
    meta = drive.files().get(fileId=txt_base.value.strip(), fields="name", supportsAllDrives=True).execute()
    buf = bajar(txt_base.value.strip())
    base = (pd.read_excel(buf, dtype=str, header=None) if meta["name"].lower().endswith(("xlsx", "xls"))
            else pd.read_csv(buf, dtype=str, header=None, sep=None, engine="python"))
    col = max(base.columns, key=lambda c: base[c].astype(str).str.fullmatch(r"\d{11}").sum())
    rucs += base[col].astype(str).str.strip().tolist()
    print(f"📥 Base de empresas: {meta['name']} (columna {col})")
rucs = list(dict.fromkeys(r for r in rucs if re.fullmatch(r"\d{11}", r)))
anios = list(range(sel_anio_ini.value, sel_anio_fin.value + 1))
if not rucs:
    raise ValueError("No hay RUC: pégalos en el Bloque 2 o indica el ID de Drive de la base de empresas.")
hechos = set(zip(ctrl_df.loc[ctrl_df["ESTADO"].isin(["OK", "SIN_DATOS"]), "RUC"],
                 ctrl_df.loc[ctrl_df["ESTADO"].isin(["OK", "SIN_DATOS"]), "AÑO"]))
pendientes = [(r, a) for r in rucs for a in anios if (r, str(a)) not in hechos]
print(f"🏢 {len(rucs)} empresa(s) × {len(anios)} año(s) = {len(rucs) * len(anios)} consultas; "
      f"pendientes: {len(pendientes)}")

# ── 4. CONSULTA POR EXPORTADOR ───────────────────────────────────────────────
MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7, "ago": 8,
         "set": 9, "sep": 9, "oct": 10, "nov": 11, "dic": 12}
diag = {"n": 0}

def diagnostico(nombre, html):
    if chk_diag.value and diag["n"] < 12:
        diag["n"] += 1
        guardar_html(f"DIAG {nombre}.html", html)

def consulta_exportador(ruc, anio):
    """Llena el formulario «Consulta por Importador/Exportador» -> HTML del resultado."""
    html, r = pedir("GET", URL_EXPORTADOR)
    if html is None:
        return None, f"formulario: {r}"
    diagnostico("1 formulario", html)
    soup = BeautifulSoup(html, "lxml")
    forms = soup.find_all("form")
    if not forms:
        return None, "la página no tiene formulario"
    form = max(forms, key=lambda f: len(f.find_all(["input", "select"])))
    datos = campos_form(form)
    for sel in form.find_all("select"):
        n, ops = sel.get("name"), opciones(sel)
        if not n:
            continue
        txt = [o[1].upper() for o in ops]
        if any("RUC" in t for t in txt):
            datos[n] = next(o[0] for o in ops if "RUC" in o[1].upper())
        elif sum(bool(re.fullmatch(r"\d{4}", o[0])) for o in ops) >= 3:
            datos[n] = str(anio)
        elif any("TODO EL A" in t for t in txt):
            datos[n] = next(o[0] for o in ops if "TODO EL A" in o[1].upper())
        elif any(t.startswith("EXPORTACI") for t in txt):
            datos[n] = next(o[0] for o in ops if o[1].upper().startswith("EXPORTACI"))
        elif any("TODAS" in t for t in txt):
            datos[n] = next(o[0] for o in ops if "TODAS" in o[1].upper())
    textos = [i["name"] for i in form.find_all("input")
              if (i.get("type") or "text").lower() == "text" and i.get("name")]
    if not textos:
        return None, "no se encontró la casilla del RUC"
    datos[textos[0]] = ruc
    for t in textos[1:]:
        datos[t] = ""
    # El botón «Consultar» suele fijar un campo oculto por JavaScript (p. ej. accion='…')
    for campo, val in re.findall(r"\.(\w+)\.value\s*=\s*['\"]([^'\"]*)['\"]", html):
        if campo in datos and not datos[campo] and "limpi" not in val.lower():
            datos[campo] = val
    pausa()
    res, r = enviar_form(form, datos, URL_EXPORTADOR)
    if res is None:
        return None, f"consulta: {r}"
    diagnostico(f"2 exportador {ruc} {anio}", res)
    return res, ""

def listar_duas(url):
    """Abre un «LISTAR» y devuelve [(aduana, año, número, fecha_embarque, canal)] con su paginación."""
    filas, total, vistas = [], None, set()
    while url and url not in vistas:
        vistas.add(url)
        pausa()
        html, r = pedir("GET", url, headers={"Referer": URL_EXPORTADOR})
        if html is None:
            return filas, total, f"listar: {r}"
        diagnostico(f"3 listar {len(vistas)}", html)
        for tr in BeautifulSoup(html, "lxml").find_all("tr"):
            c = [limpio(td.get_text(" ")) for td in tr.find_all("td", recursive=False)]
            if (len(c) >= 4 and re.fullmatch(r"\d{3}", c[0]) and re.fullmatch(r"\d{4}", c[1])
                    and re.fullmatch(r"\d{4,6}", c[2])):
                filas.append((c[0], c[1], c[2].zfill(6), c[3] if len(c) > 3 else "", c[4] if len(c) > 4 else ""))
        hasta, total = paginacion(html)
        url = siguiente_pagina(html, url) if total and hasta and hasta < total else None
    return filas, total, ""

# ── 5. RECORRIDO ─────────────────────────────────────────────────────────────
nuevas_exp, nuevas_ind = [], []
ind_ids = set(ind_df["DUA_ID"])
for k, (ruc, anio) in enumerate(pendientes, 1):
    html, err = consulta_exportador(ruc, anio)
    ctrl = {"RUC": ruc, "AÑO": str(anio), "FECHA_EXTRACCION": HOY}
    if html is None:
        ctrl.update(ESTADO="ERROR", NOTA=err)
    else:
        links = links_con(html, "buscarListadoDuas", URL_EXPORTADOR)
        hasta, total = paginacion(html)
        if not links:
            sin = re.search(r"no (existe|se encontr|hay)|sin registros|no registra", html, re.I)
            ctrl.update(ESTADO="SIN_DATOS" if sin else "REVISAR",
                        NOTA="" if sin else "no se reconocieron filas con LISTAR (ver HTML de diagnóstico)",
                        FILAS_MES="0", DUAS="0")
            if not sin:
                guardar_html(f"REVISAR exportador {ruc} {anio}.html", html)
        else:
            fob, ok, n_duas = 0.0, 0, 0
            for url, celdas in links:
                q = {kk: v[0] for kk, v in parse_qs(urlparse(url).query).items()}
                def c(i, celdas=celdas):        # columnas desde la derecha: …, MES, AGENTE, ADUANA, PAÍS, FOB
                    return celdas[i] if len(celdas) >= -i else ""
                fob_mes = numero(c(-1))
                fob += fob_mes or 0
                duas, tot, err_l = listar_duas(url)
                ok += not err_l and (tot is None or len(duas) >= tot)
                n_duas += len(duas)
                mes_txt = c(-5)
                nuevas_exp.append({"RUC": ruc, "EXPORTADOR": c(-6), "AÑO": str(anio),
                                   "MES": q.get("CG_mes") or str(MESES.get(mes_txt[:3].lower(), "")),
                                   "MES_TXT": mes_txt, "AGENTE": c(-4),
                                   "ADUANA": c(-3), "PAIS": c(-2),
                                   "FOB_USD": fob_mes, "ADUANA_COD": q.get("CG_aduana", ""),
                                   "AGENTE_COD": q.get("CG_agente", ""), "PAIS_COD": q.get("CG_pais", ""),
                                   "URL_LISTAR": url, "DUAS_LISTADAS": len(duas), "DUAS_TOTAL": tot,
                                   "FECHA_EXTRACCION": HOY})
                for ad, an, num, fecha, canal in duas:
                    did = f"{ad}-{an}-{REGIMEN}-{num}"
                    if did in ind_ids:
                        continue
                    ind_ids.add(did)
                    nuevas_ind.append({"DUA_ID": did, "RUC": ruc, "AÑO": str(anio), "MES": q.get("CG_mes", ""),
                                       "ADUANA_COD": ad, "AGENTE_COD": q.get("CG_agente", ""),
                                       "PAIS_COD": q.get("CG_pais", ""), "ANIO_DUA": an, "NUMERO": num,
                                       "FECHA_EMBARQUE": fecha, "CANAL": canal, "ESTADO_DETALLE": "PENDIENTE",
                                       "FECHA_EXTRACCION": HOY})
            completo = total is None or len(links) >= total
            ctrl.update(FILAS_MES=str(len(links)), FOB_USD_MES=f"{fob:.2f}", LISTADOS_OK=str(ok),
                        DUAS=str(n_duas),
                        ESTADO="OK" if ok == len(links) and completo else "REVISAR",
                        NOTA="" if completo else f"la tabla muestra {hasta} de {total} filas")
    ctrl_df = pd.concat([ctrl_df[~((ctrl_df["RUC"] == ruc) & (ctrl_df["AÑO"] == str(anio)))],
                         pd.DataFrame([ctrl])], ignore_index=True)
    print(f"[{k}/{len(pendientes)}] {ruc} {anio}: {ctrl['ESTADO']} · filas {ctrl.get('FILAS_MES', '-')}"
          f" · DUA {ctrl.get('DUAS', '-')} {ctrl.get('NOTA', '')}")
    if k % 5 == 0 or k == len(pendientes):         # guarda cada 5 consultas y al final
        exp_df = pd.concat([exp_df[~exp_df.set_index(["RUC", "AÑO"]).index.isin(
                               {(e["RUC"], e["AÑO"]) for e in nuevas_exp})], pd.DataFrame(nuevas_exp)],
                           ignore_index=True)[COLS_EXP]
        ind_df = pd.concat([ind_df, pd.DataFrame(nuevas_ind)], ignore_index=True)[COLS_IND].fillna("")
        nuevas_exp, nuevas_ind = [], []
        guardar(exp_df, N_EXPORTADOR, F_DATOS, datos_drive)
        guardar(ind_df, N_INDICE, F_DATOS, datos_drive)
        guardar(ctrl_df[COLS_CTRL].fillna(""), N_CONTROL, DRIVE_FOLDER_CONTROL_ID, control_drive)
        subir_html("fase A")
    pausa()

pend_b = (ind_df["ESTADO_DETALLE"] != "OK").sum()
print(f"\n✅ Fase A terminada. Control: {ctrl_df['ESTADO'].value_counts().to_dict()}")
print(f"   DUA en el índice: {len(ind_df)} · pendientes de detalle (Bloque 4): {pend_b}")

In [ ]:
# ═══ BLOQUE 4 · FASE B: DETALLE DE CADA DUA (te pide el captcha) ═══
# Abre cada DUA pendiente del índice. Cuando SUNAT muestra el captcha, el notebook enseña la imagen y
# tú escribes el código (Enter vacío = otra imagen · s = saltar esta DUA · q = terminar y guardar).
# Luego baja la cabecera y TODAS las series sin pedir nada más. Guarda cada 10 DUA; si se corta,
# vuelve a ejecutar este bloque y sigue con las pendientes.
COLS_CAB = ["DUA_ID", "RUC", "EXPORTADOR", "DIRECCION", "DISTRITO_DIR", "PROVINCIA_DIR", "DEPARTAMENTO_DIR",
            "UBIGEO", "ADUANA", "ADUANA_COD", "AÑO_DUA", "NUMERO", "FECHA_NUMERACION", "CANAL", "N_ORDEN",
            "DESTINACION", "MODALIDAD", "TIPO_DESPACHO", "EMPRESA_TRANSPORTE", "MANIFIESTO", "VIA_TRANSPORTE",
            "DEPOSITO_TEMPORAL", "MODALIDAD_PAGO", "VALOR_VENTA_USD", "FOB_USD", "DECLARANTE",
            "PESO_NETO_KG", "PESO_BRUTO_KG", "BULTOS", "U_FISICAS", "U_COMERCIALES", "N_SERIES",
            "FOB_SERIES_USD", "ESTADO", "FECHA_EXTRACCION", "FICHA_JSON"]
COLS_SER = ["DUA_ID", "RUC", "SERIE", "PARTIDA", "FOB_USD", "PESO_NETO_KG", "PESO_BRUTO_KG", "BULTOS", "CLASE",
            "CANT_FISICA", "UNIDAD_FISICA", "CANT_COMERCIAL", "PUERTO_EMBARQUE", "FECHA_EMBARQUE",
            "DOC_TRANSPORTE", "PAIS_ORIGEN", "PAIS_DESTINO", "TPN", "REG_APLICACION", "FLETE_USD", "SEGURO_USD",
            "VALOR_ADUANA_USD", "DESC_1", "DESC_2", "DESC_3", "DESC_4", "DESC_5", "FECHA_EXTRACCION", "FICHA_JSON"]
cab_df = leer_previo(N_CABECERA, datos_drive, COLS_CAB)
ser_df = leer_previo(N_SERIES, datos_drive, COLS_SER)
cola = ind_df[ind_df["ESTADO_DETALLE"] != "OK"].sort_values(["RUC", "AÑO", "ADUANA_COD", "ANIO_DUA", "NUMERO"])
cola = cola.head(txt_max_duas.value)
print(f"🧾 DUA pendientes en el índice: {(ind_df['ESTADO_DETALLE'] != 'OK').sum()} · en esta corrida: {len(cola)}")

def es_dua(html):
    h = (html or "").upper()
    return "IDENTIFICACION" in h and ("DECLARACION UNICA" in h or "REGISTRO DE ADUANA" in h)

def imagen_captcha(soup, url):
    """Bytes de la imagen del captcha (la <img> del formulario que no es el logo)."""
    imgs = [i for i in soup.find_all("img") if i.get("src")]
    imgs = sorted(imgs, key=lambda i: (not re.search(r"captcha|imagen|codigo|kaptcha|jcaptcha|servlet", i["src"], re.I),
                                       bool(re.search(r"logo|sunat|\.gif$", i["src"], re.I))))
    for i in imgs:
        try:
            r = sesion.get(urljoin(url, i["src"]), headers={"Referer": url}, timeout=60)
        except requests.RequestException:
            continue
        if r.status_code == 200 and r.headers.get("Content-Type", "").startswith("image"):
            return r.content
    return None

def abrir_dua(ad, an, num):
    """-> (html, url, estado). estado: '' (OK), 'SALTAR', 'TERMINAR' o un error."""
    url = f"{URL_DUA}?option=una&n={REGIMEN}&codaduana={ad}&anoprese={an}&numecorre={num}"
    html, r = pedir("GET", url)
    for _ in range(8):
        if html is None:
            return None, url, f"DUA: {r}"
        if es_dua(html):
            return html, url, ""
        soup = BeautifulSoup(html, "lxml")
        form = next((f for f in soup.find_all("form")
                     if any((i.get("type") or "text").lower() == "text" for i in f.find_all("input"))), None)
        if form is None:
            guardar_html(f"REVISAR dua {ad}-{an}-{num}.html", html)
            return None, url, "sin formulario de captcha ni datos de la DUA"
        img = imagen_captcha(soup, url)
        if img:
            display(Image(data=img))
        else:
            print("   (no se pudo mostrar la imagen; ábrela en el navegador)")
        cod = input(f"DUA {ad}-{an}-{REGIMEN}-{num} · captcha (Enter = otra imagen, s = saltar, q = terminar): ").strip()
        if cod.lower() == "q":
            return None, url, "TERMINAR"
        if cod.lower() == "s":
            return None, url, "SALTAR"
        if not cod:
            html, r = pedir("GET", url)
            continue
        datos = campos_form(form)
        caja_txt = [i["name"] for i in form.find_all("input")
                    if (i.get("type") or "text").lower() == "text" and i.get("name")]
        datos[caja_txt[0]] = cod
        html, r = enviar_form(form, datos, url)
        if html is not None and not es_dua(html):
            print("   ✗ El código no fue aceptado; prueba otra vez.")
    return None, url, "captcha no aceptado tras varios intentos"

def series_de(html):
    """(form, nombre_select, [números de serie]) del desplegable «SERIE»."""
    soup = BeautifulSoup(html, "lxml")
    for sel in soup.find_all("select"):
        nums = [v for v, t in opciones(sel) if re.fullmatch(r"\d{1,4}", v)]
        if nums:
            return sel.find_parent("form"), sel.get("name"), nums
    return None, None, []

def abrir_serie(html_dua, url_dua, ad, an, num, s):
    """HTML de la serie `s`: con la plantilla del Bloque 2, con el formulario del desplegable o con el enlace del JS."""
    if txt_plantilla.value.strip():
        u = urlparse(urljoin(BASE, txt_plantilla.value.strip()))
        q = {k: v[0] for k, v in parse_qs(u.query, keep_blank_values=True).items()}
        viejo = {"ad": q.get("codaduana"), "an": q.get("anoprese"), "num": q.get("numecorre")}
        for k, v in q.items():
            if "serie" in k.lower():
                q[k] = s
            elif v == viejo["ad"]:
                q[k] = ad
            elif v == viejo["an"]:
                q[k] = an
            elif v == viejo["num"]:
                q[k] = num
        return pedir("GET", u._replace(query=urlencode(q)).geturl(), headers={"Referer": url_dua})
    form, nombre, _ = series_de(html_dua)
    if form is not None and nombre:
        datos = campos_form(form)
        datos[nombre] = s
        return enviar_form(form, datos, url_dua)
    m = re.search(r"['\"]([^'\"]*SgDetSerie\?[^'\"]*=)['\"]", html_dua)
    if m:
        return pedir("GET", urljoin(url_dua, m.group(1) + s), headers={"Referer": url_dua})
    return None, "no se encontró cómo abrir las series (pega en el Bloque 2 la URL de una serie)"

def partes_direccion(d):
    p = [x.strip() for x in str(d or "").split(",") if x.strip()]
    return (p[-3] if len(p) >= 3 else "", p[-2] if len(p) >= 2 else "", p[-1] if p else "")

nuevas_cab, nuevas_ser, hechas, terminar = [], [], 0, False
def volcar():
    global cab_df, ser_df, nuevas_cab, nuevas_ser
    if nuevas_cab:
        ids = {c["DUA_ID"] for c in nuevas_cab}
        cab_df = pd.concat([cab_df[~cab_df["DUA_ID"].isin(ids)], pd.DataFrame(nuevas_cab)], ignore_index=True)[COLS_CAB]
        ser_df = pd.concat([ser_df[~ser_df["DUA_ID"].isin(ids)], pd.DataFrame(nuevas_ser)], ignore_index=True)[COLS_SER]
        nuevas_cab, nuevas_ser = [], []
    guardar(cab_df.fillna(""), N_CABECERA, F_DATOS, datos_drive)
    guardar(ser_df.fillna(""), N_SERIES, F_DATOS, datos_drive)
    guardar(ind_df.fillna(""), N_INDICE, F_DATOS, datos_drive)
    subir_html("fase B")

for k, fila in enumerate(cola.itertuples(index=False), 1):
    did, ad, an, num = fila.DUA_ID, fila.ADUANA_COD, fila.ANIO_DUA, fila.NUMERO
    html, url, err = abrir_dua(ad, an, num)
    i_ind = ind_df.index[ind_df["DUA_ID"] == did]
    if err == "TERMINAR":
        terminar = True
        break
    if html is None:
        ind_df.loc[i_ind, ["ESTADO_DETALLE", "NOTA", "FECHA_DETALLE"]] = [
            "SALTADA" if err == "SALTAR" else "ERROR", err, HOY]
        continue
    guardar_html(f"{did} cabecera.html", html)
    f = leer_ficha(html)
    doc = valor(f, "1.2")
    ruc = (re.findall(r"\d{11}", doc) or [fila.RUC])[0]
    direc = valor(f, "1.3")
    dist, prov, dep = partes_direccion(direc)
    _, _, nums = series_de(html)
    fob_ser, errores_s = 0.0, []
    for s in nums:
        pausa()
        hs, r = abrir_serie(html, url, ad, an, num, s)
        if hs is None:
            errores_s.append(f"serie {s}: {r}")
            continue
        guardar_html(f"{did} serie {s}.html", hs)
        fs = leer_ficha(hs)
        fob = numero(valor(fs, "7.30"))
        fob_ser += fob or 0
        cant = valor(fs, "7.16")
        desc = descripcion_serie(hs)
        nuevas_ser.append({"DUA_ID": did, "RUC": ruc, "SERIE": s,
                           "PARTIDA": re.sub(r"\D", "", valor(fs, "7.19"))[:10],
                           "FOB_USD": fob, "PESO_NETO_KG": numero(valor(fs, "7.14")),
                           "PESO_BRUTO_KG": numero(valor(fs, "7.15")), "BULTOS": numero(valor(fs, "7.12")),
                           "CLASE": valor(fs, "7.13"), "CANT_FISICA": numero(cant),
                           "UNIDAD_FISICA": cant.split("-")[-1].strip() if "-" in cant else "",
                           "CANT_COMERCIAL": numero(valor(fs, "7.10")), "PUERTO_EMBARQUE": valor(fs, "7.6"),
                           "FECHA_EMBARQUE": valor(fs, "7.7"), "DOC_TRANSPORTE": valor(fs, "7.8"),
                           "PAIS_ORIGEN": valor(fs, "7.26"), "PAIS_DESTINO": valor(fs, "7.27"),
                           "TPN": valor(fs, "7.24"), "REG_APLICACION": valor(fs, "7.28"),
                           "FLETE_USD": numero(valor(fs, "7.31")), "SEGURO_USD": numero(valor(fs, "7.32")),
                           "VALOR_ADUANA_USD": numero(valor(fs, "7.34")),
                           **{f"DESC_{i}": d for i, d in enumerate(desc, 1)},
                           "FECHA_EXTRACCION": HOY, "FICHA_JSON": json.dumps(fs, ensure_ascii=False)})
    fob_dua = numero(valor(f, "6.5"))
    ok = nums and not errores_s and fob_dua is not None and abs(fob_dua - fob_ser) <= 1
    estado = "OK" if ok else "REVISAR"
    nuevas_cab.append({"DUA_ID": did, "RUC": ruc, "EXPORTADOR": valor(f, "1.1"), "DIRECCION": direc,
                       "DISTRITO_DIR": dist, "PROVINCIA_DIR": prov, "DEPARTAMENTO_DIR": dep, "UBIGEO": valor(f, "1.4"),
                       "ADUANA": f.get("Aduana", ""), "ADUANA_COD": ad, "AÑO_DUA": an, "NUMERO": num,
                       "FECHA_NUMERACION": f.get("Fecha Numeración", ""), "CANAL": f.get("Sujeto a", ""),
                       "N_ORDEN": f.get("Nº Orden", ""), "DESTINACION": f.get("Destinación", ""),
                       "MODALIDAD": f.get("Modalidad", ""), "TIPO_DESPACHO": f.get("Tipo Despacho", ""),
                       "EMPRESA_TRANSPORTE": valor(f, "3.1"), "MANIFIESTO": valor(f, "3.2"),
                       "VIA_TRANSPORTE": valor(f, "3.3"), "DEPOSITO_TEMPORAL": valor(f, "4.1"),
                       "MODALIDAD_PAGO": valor(f, "5.2"), "VALOR_VENTA_USD": numero(valor(f, "6.1")),
                       "FOB_USD": fob_dua, "DECLARANTE": valor(f, "8.1"),
                       "PESO_NETO_KG": numero(f.get("Total Peso Neto")), "PESO_BRUTO_KG": numero(f.get("Total Peso Bruto")),
                       "BULTOS": numero(f.get("Total Bultos")),
                       "U_FISICAS": numero(next((v for kk, v in f.items() if kk.startswith("Total U.F")), "")),
                       "U_COMERCIALES": numero(next((v for kk, v in f.items() if kk.startswith("Total U.C")), "")),
                       "N_SERIES": len(nums), "FOB_SERIES_USD": round(fob_ser, 2), "ESTADO": estado,
                       "FECHA_EXTRACCION": HOY, "FICHA_JSON": json.dumps(f, ensure_ascii=False)})
    ind_df.loc[i_ind, ["ESTADO_DETALLE", "N_SERIES", "FOB_DUA", "FOB_SERIES", "NOTA", "FECHA_DETALLE"]] = [
        estado, str(len(nums)), "" if fob_dua is None else f"{fob_dua:.2f}", f"{fob_ser:.2f}",
        "; ".join(errores_s) or ("" if ok else "FOB de las series ≠ FOB de la DUA" if nums else "sin series"), HOY]
    hechas += 1
    print(f"   ✓ [{k}/{len(cola)}] {did} · {valor(f, '1.1')[:40]} · {len(nums)} serie(s) · FOB {fob_dua} · {estado}")
    if hechas % 10 == 0:
        volcar()
    pausa()

volcar()
print(f"\n✅ Fase B: {hechas} DUA leídas{' (terminada a pedido)' if terminar else ''}. "
      f"Índice: {ind_df['ESTADO_DETALLE'].value_counts().to_dict()}")